# 4.2 Model — Random Forest

This notebook builds the second model of the project, a **random forest**, with the same protocol as the logistic-regression baseline (`1_BaselineModel.ipynb`): the same 7 temporal splits, the same recipe and the same feature-selection strategy, so the two models can be compared split by split.

- **Assessment:** inside each split, the recipe, the feature selection, the forest and the cut-off are fitted on the training rows only; the validation rows are only scored.
- **Cut-off:** a forest predicts its own training rows almost perfectly, so the cut-off is chosen on the **out-of-bag** probabilities (each row scored only by the trees that did not see it), not on the training probabilities used for the logistic regression.
- **Against the baseline:** paired comparison with the logistic regression on the same splits.
- **Tuning:** a random search of the forest's hyperparameters, scored on splits that took no part in the choice (walk-forward).
- **Final model:** fitted once on every training row, writing the Kaggle submission.

## 4.2.1 Setup

In [1]:
# Standard library
import json
import sys
import warnings
from pathlib import Path

# Arrays, tables, the model and its metrics
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.ensemble import RandomForestClassifier
from sklearn.exceptions import ConvergenceWarning
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (average_precision_score, balanced_accuracy_score, confusion_matrix,
                             f1_score, precision_score, recall_score, roc_auc_score)
from sklearn.model_selection import ParameterSampler

# The group's own functions (src/3_Feature_selection/preprocessing.py and selection.py)
sys.path.append("../../../src/3_Feature_selection")
from preprocessing import (RANDOM_STATE, fold_matrices, load_data, paired_delta, purchase_dates,
                           target, temporal_splits)
from selection import make_strategy_selector

# Functions of the model notebooks (src/4_Model/model.py)
sys.path.append("../../../src/4_Model")
from model import (CUTOFFS, baseline_rule, best_cutoff, f1_by_cutoff, fit_forest_with_cutoff,
                   fit_with_cutoff, score_configurations, walk_forward)

warnings.filterwarnings("ignore", category=UserWarning)  # unknown levels in validation, as in feature selection

In [2]:
# The feature-selection strategy: the candidate columns and the selector's settings
strategy = json.loads(Path("../../../logs/feature_selection.json").read_text(encoding="utf-8"))
numeric = strategy["candidate_columns"]["numeric"]
categorical = strategy["candidate_columns"]["categorical"]
selector = strategy["selector"]

train_clientes = load_data()
# The splits need the raw date parts, so they are built before the calendar columns are
# replaced by their sin/cos components, as in the feature-selection notebook
time_splits = temporal_splits(train_clientes)
for col, period in [("purchase_month", 12.0), ("purchase_day", 31.0), ("purchase_dayofweek", 7.0)]:
    angle = 2.0 * np.pi * train_clientes[col].astype(float) / period
    train_clientes[f"{col}_sin"] = np.sin(angle)
    train_clientes[f"{col}_cos"] = np.cos(angle)
frame = train_clientes[numeric + categorical + [target]]

# The pre-processing team's rules (1.6, 1.7.3, 1.7.4 and 1.8), as in the baseline notebook
labels_as_numbers = ["is_first_purchase", "is_weekend", "since_2015_or_earlier", "purchase_year",
                     "customer_since_year", "purchase_month_sin", "purchase_month_cos",
                     "purchase_day_sin", "purchase_day_cos", "purchase_dayofweek_sin",
                     "purchase_dayofweek_cos"]
recipe = {"numeric": numeric, "categorical": categorical,
          "fill": [(numeric, SimpleImputer(strategy="median")),
                   (categorical, SimpleImputer(strategy="most_frequent"))],
          "log1p": [c for c in numeric
                    if c not in labels_as_numbers and not (train_clientes[c] < 0).any()],
          "rare": {"transport_method": 50, "payment_type": 20, "city": 30, "municipality": 30,
                   "postal_area": 20}}
print(f"{len(frame):,} training rows | {len(numeric)} numeric and {len(categorical)} categorical"
      f" candidate columns | {len(time_splits)} temporal splits")

6,469 training rows | 39 numeric and 8 categorical candidate columns | 7 temporal splits


## 4.2.2 Assessment on the Temporal Splits

A random forest does not need scaled columns or a penalty to cope with correlated ones, so the feature selection is no longer a necessity. Two versions of the forest are assessed on the same splits, to see whether the selection still helps:

- **all columns:** every column the recipe produces;
- **selected columns:** the columns kept by the strategy of `logs/feature_selection.json` (multicollinearity filter at 0.90, then L1), fitted inside each split.

The first settings are a reasonable starting point, not tuned: 300 trees, `min_samples_leaf=5` (leaves of a single row would memorise the training rows) and `max_features="sqrt"`. The tuning of 4.5 searches around them.

In [3]:
def make_forest(**settings):
    """A fresh, unfitted random forest. `oob_score=True` gives the out-of-bag probabilities that
    choose the cut-off; the defaults are the starting point of 4.2."""
    params = {"n_estimators": 300, "min_samples_leaf": 5, "max_features": "sqrt"}
    params.update(settings)
    return RandomForestClassifier(oob_score=True, n_jobs=-1, random_state=RANDOM_STATE, **params)


def assess(use_selection):
    """The 7 splits for one version of the forest: its measures per split and the matrices it used."""
    rows, pooled, prepared = [], [], []
    dates = purchase_dates(train_clientes)
    for train_rows, validation_rows in time_splits:
        train, validation = frame.iloc[train_rows], frame.iloc[validation_rows]
        A_train, A_validation, names = fold_matrices(train, validation, recipe)
        keep = np.ones(len(names), dtype=bool)
        if use_selection:
            keep = make_strategy_selector(selector).fit(A_train, train[target]).get_support()
        prepared.append((A_train[:, keep], A_validation[:, keep], train[target], validation[target], names[keep]))

        # The cut-off is chosen on the out-of-bag probabilities of the training rows
        model, p_oob, cutoff = fit_forest_with_cutoff(make_forest(), A_train[:, keep], train[target])
        p_validation = model.predict_proba(A_validation[:, keep])[:, 1]
        y, predicted = validation[target], (p_validation >= cutoff).astype(int)
        rows.append({"validation from": dates.iloc[validation_rows].min().date(),
                     "until": dates.iloc[validation_rows].max().date(),
                     "columns": int(keep.sum()), "cut-off": cutoff,
                     "OOB F1": f1_score(train[target], p_oob >= cutoff),
                     "F1": f1_score(y, predicted),
                     "AP": average_precision_score(y, p_validation),
                     "ROC-AUC": roc_auc_score(y, p_validation),
                     "balanced accuracy": balanced_accuracy_score(y, predicted),
                     "precision": precision_score(y, predicted),
                     "recall": recall_score(y, predicted)})
        pooled.append(pd.DataFrame({"actual": y.to_numpy(), "predicted": predicted}))
    return pd.DataFrame(rows), pooled, prepared


measures = ["OOB F1", "F1", "AP", "ROC-AUC", "balanced accuracy", "precision", "recall"]
runs = {"all columns": assess(use_selection=False), "selected columns": assess(use_selection=True)}
for name, (table, _, _) in runs.items():
    print(f"=== forest on {name}")
    print(table[["validation from", "until", "columns", "cut-off"] + measures].round(3).to_string(index=False))
    print("\nmean:", table[measures].mean().round(4).to_dict())
    print("SEM: ", (table[measures].std(ddof=1) / np.sqrt(len(table))).round(4).to_dict(), "\n")

=== forest on all columns
validation from      until  columns  cut-off  OOB F1    F1    AP  ROC-AUC  balanced accuracy  precision  recall
     2019-06-09 2020-03-28       83     0.39   0.756 0.744 0.854    0.783              0.644      0.660   0.852
     2020-03-28 2021-03-06       84     0.38   0.752 0.754 0.838    0.745              0.626      0.685   0.839
     2021-03-06 2021-11-13       90     0.40   0.755 0.770 0.863    0.829              0.693      0.669   0.908
     2021-11-13 2022-10-02       93     0.37   0.755 0.711 0.828    0.803              0.615      0.571   0.942
     2022-10-04 2023-07-28      103     0.39   0.756 0.730 0.822    0.805              0.669      0.619   0.888
     2023-07-29 2024-04-13      106     0.41   0.754 0.789 0.877    0.800              0.671      0.733   0.855
     2024-04-13 2025-02-15      106     0.40   0.759 0.806 0.874    0.821              0.680      0.715   0.924

mean: {'OOB F1': 0.7552, 'F1': 0.7579, 'AP': 0.8509, 'ROC-AUC': 0.7978, 'bala

**Conclusions:**
- Over the 7 splits the forest on the selected columns reaches **F1 0.763 ± 0.014** (mean ± SEM) at its cut-off, with AP 0.849, ROC-AUC 0.798 and balanced accuracy 0.678. Its out-of-bag F1 on the training rows (0.757) stays close to the validation F1, so the forest is not overfitting and the cut-off chosen on the out-of-bag probabilities carries over to new rows.
- F1 depends on the period: from 0.723 in the block starting in November 2021 to 0.823 in the latest block (April 2024 to February 2025), the one closest to the test period.
- The cut-off is stable, between 0.38 and 0.43, and the feature selection keeps between 23 and 37 columns as the training rows grow. Without the selection (83 to 106 columns) the forest scores AP 0.851 and F1 0.758; 4.3 compares the two.

## 4.2.3 Against the Logistic Regression and the Floors

The forest is compared, split by split, with the logistic regression of the baseline notebook (same recipe, same selection, cut-off chosen on the training rows), with the competition's baseline rule (`purchase_count_365d >= 3`) and with always predicting a repeat purchase. As in the other notebooks, **AP decides** and F1 is shown beside it; a difference counts only when it is larger than two standard errors.

In [4]:
dates = purchase_dates(train_clientes)
logistic, floors = [], []
for (A_train, A_validation, y_train, y_validation, _), (_, validation_rows) in zip(runs["selected columns"][2], time_splits):
    model, _, cutoff = fit_with_cutoff(LogisticRegression(max_iter=1000, random_state=RANDOM_STATE), A_train, y_train)
    p = model.predict_proba(A_validation)[:, 1]
    logistic.append({"AP": average_precision_score(y_validation, p), "F1": f1_score(y_validation, p >= cutoff)})
    validation = frame.iloc[validation_rows]
    floors.append({"always 1": f1_score(y_validation, np.ones(len(y_validation), dtype=int)),
                   "baseline rule": f1_score(y_validation, baseline_rule(validation))})
logistic, floors = pd.DataFrame(logistic), pd.DataFrame(floors)

forest_all, forest_selected = runs["all columns"][0], runs["selected columns"][0]
summary = pd.DataFrame({
    "logistic regression": {"AP": logistic["AP"].mean(), "F1": logistic["F1"].mean()},
    "forest, all columns": {"AP": forest_all["AP"].mean(), "F1": forest_all["F1"].mean()},
    "forest, selected columns": {"AP": forest_selected["AP"].mean(), "F1": forest_selected["F1"].mean()},
    "baseline rule": {"AP": np.nan, "F1": floors["baseline rule"].mean()},
    "always 1": {"AP": np.nan, "F1": floors["always 1"].mean()}}).T
print(summary.round(4).to_string(), "\n")

def compare(label, a, b, metric):
    r = paired_delta(a, b)
    print(f"{label:55s} d{metric} {r['delta']:+.4f} +/- {r['SEM']:.4f}, better on {r['wins']} -> {r['verdict']}")

for metric in ("AP", "F1"):
    compare("forest (selected) minus logistic regression", forest_selected[metric], logistic[metric], metric)
    compare("forest (all columns) minus logistic regression", forest_all[metric], logistic[metric], metric)
    compare("forest (all columns) minus forest (selected)", forest_all[metric], forest_selected[metric], metric)
compare("forest (selected) minus baseline rule", forest_selected["F1"], floors["baseline rule"], "F1")

                              AP      F1
logistic regression       0.8421  0.7543
forest, all columns       0.8509  0.7579
forest, selected columns  0.8491  0.7634
baseline rule                NaN  0.7461
always 1                     NaN  0.7186 

forest (selected) minus logistic regression             dAP +0.0070 +/- 0.0022, better on 7/7 -> better
forest (all columns) minus logistic regression          dAP +0.0088 +/- 0.0023, better on 7/7 -> better
forest (all columns) minus forest (selected)            dAP +0.0018 +/- 0.0020, better on 6/7 -> inside the noise
forest (selected) minus logistic regression             dF1 +0.0092 +/- 0.0031, better on 7/7 -> better
forest (all columns) minus logistic regression          dF1 +0.0037 +/- 0.0040, better on 5/7 -> inside the noise
forest (all columns) minus forest (selected)            dF1 -0.0055 +/- 0.0028, better on 1/7 -> inside the noise
forest (selected) minus baseline rule                   dF1 +0.0173 +/- 0.0052, better on 7/7 -> b

**Conclusions:**
- **AP:** the forest ranks the customers better than the logistic regression, 0.849 against 0.842 (**+0.0070 ± 0.0022**), ahead on all 7 splits and by more than two standard errors. The gap is small next to the differences between periods (AP from 0.819 to 0.877 across the splits).
- **F1:** +0.0092 ± 0.0031 over the logistic regression (0.763 against 0.754), also ahead on 7 of 7. Against the baseline rule (0.746) it is **+0.0173 ± 0.0052**, ahead on 7 of 7: the gap of the logistic regression to the rule (+0.008) was inside the noise, this one is not. Always 1 scores 0.719.
- **The selection:** with all columns the forest is +0.0018 ± 0.0020 in AP and −0.0055 ± 0.0028 in F1 against the selected columns, both inside the noise, although the F1 difference sits close to the limit. The forest does not need the selection, so the final model keeps the 40 selected columns, which the feature-selection tables already hold.

## 4.2.4 Reading the Errors and the Importances

The confusion matrix gathers the predictions of the 7 validation blocks. The importance of each column is the mean decrease in impurity of a forest fitted on each split's training rows (selected columns), averaged over the splits where the column was kept.

In [5]:
both = pd.concat(runs["selected columns"][1], ignore_index=True)
tn, fp, fn, tp = confusion_matrix(both["actual"], both["predicted"]).ravel()
print(f"{len(both):,} validation purchases: {tp} repeat purchases caught, {fn} missed,"
      f" {fp} false alarms, {tn} correctly left out\n")

importances = []
for A_train, _, y_train, _, names in runs["selected columns"][2]:
    importances.append(pd.Series(make_forest().fit(A_train, y_train).feature_importances_, index=names))
importances = pd.concat(importances, axis=1)
top = pd.DataFrame({"mean importance": importances.mean(axis=1),
                    "kept in": importances.notna().sum(axis=1).astype(str) + f"/{importances.shape[1]}"})
print(top.sort_values("mean importance", ascending=False).head(12).round(4).to_string())

4,116 validation purchases: 2003 repeat purchases caught, 311 missed, 924 false alarms, 878 correctly left out

                              mean importance kept in
purchase_count_365d                    0.0929     6/7
spend_365d                             0.0895     6/7
spend_180d                             0.0734     5/7
purchase_count_90d                     0.0639     7/7
purchase_count_180d                    0.0579     1/7
prior_mean_gap_days                    0.0570     7/7
prior_max_purchase_value               0.0558     3/7
Total Bruto                            0.0519     3/7
days_since_previous_purchase           0.0507     7/7
prior_total_spend                      0.0501     2/7
prior_purchase_count                   0.0489     5/7
Total Desc_ Linha                      0.0468     7/7


**Conclusions:**
- Of the 2,314 repeat purchases in the validation blocks, the forest catches 2,003 (recall 0.87) and misses 311. Of its 2,927 positive predictions, 924 are false alarms (precision 0.68). Against the logistic regression (1,857 caught, 744 false alarms) it catches 146 more repeat purchases and raises 180 more false alarms.
- The columns that matter most describe the customer's recent activity: the counts and spend over the last 365, 180 and 90 days lead (`purchase_count_365d` 0.093, `spend_365d` 0.090, `spend_180d` 0.073, `purchase_count_90d` 0.064), followed by the customer's history (`prior_mean_gap_days`, `prior_max_purchase_value`, `days_since_previous_purchase`). This agrees with the baseline rule, which uses `purchase_count_365d`.
- The importance is spread over many columns, and correlated columns share it (`purchase_count_180d` is kept in only 1 of the 7 splits), so the ranking is a rough guide and not a measure of the effect of each column.

## 4.2.5 Tuning the Random Forest

The forest has several settings that control how closely it fits the training rows. A random search tries combinations of:

- **`max_depth`:** none, 6, 10 or 15;
- **`min_samples_leaf`:** 1, 3, 5, 10 or 20;
- **`max_features`:** `"sqrt"`, `"log2"`, 0.3 or 0.5 of the columns at each split;
- **`class_weight`:** none, or `"balanced_subsample"`;
- **`max_samples`:** every row in each tree's bootstrap, or 70%.

The search runs on the same 7 splits and on the matrices of 4.2 (selected columns), plus the starting point of 4.2, with 150 trees each to keep the search affordable (the final model of 4.6 uses more). Each combination chooses its own cut-off on its out-of-bag probabilities. AP decides. As for the logistic regression, the search is then **scored on splits that took no part in the choice** (walk-forward) and is adopted only if it beats the starting point by more than two standard errors.

In [6]:
space = {"max_depth": [None, 6, 10, 15], "min_samples_leaf": [1, 3, 5, 10, 20],
         "max_features": ["sqrt", "log2", 0.3, 0.5], "class_weight": [None, "balanced_subsample"],
         "max_samples": [None, 0.7]}
starting_point = {"max_depth": None, "min_samples_leaf": 5, "max_features": "sqrt",
                  "class_weight": None, "max_samples": None}
sampled = list(ParameterSampler(space, n_iter=10, random_state=RANDOM_STATE))
grid = [starting_point] + [c for c in sampled if c != starting_point]
default = 0


def config_name(index):
    return ", ".join(f"{key}={value}" for key, value in grid[index].items())


rows = []
for index, config in enumerate(grid):
    for split, (A_train, A_validation, y_train, y_validation, _) in enumerate(runs["selected columns"][2]):
        model, _, cutoff = fit_forest_with_cutoff(make_forest(**{**config, "n_estimators": 150}), A_train, y_train)
        p = model.predict_proba(A_validation)[:, 1]
        rows.append({"configuration": index, "split": split, "cut-off": cutoff,
                     "AP": average_precision_score(y_validation, p), "F1": f1_score(y_validation, p >= cutoff)})
scores = pd.DataFrame(rows)
mean_ap = scores.groupby("configuration")["AP"].mean()

reference_run = scores[scores["configuration"] == default].set_index("split")
board = {}
for index in dict.fromkeys(list(mean_ap.nlargest(5).index) + [default]):
    run = scores[scores["configuration"] == index].set_index("split")
    r = paired_delta(run["AP"], reference_run["AP"]) if index != default else {"delta": 0.0, "SEM": 0.0, "wins": "0/7", "verdict": "-"}
    board[config_name(index) + (" (starting point)" if index == default else "")] = {
        "AP": run["AP"].mean(), "dAP": r["delta"], "AP better on": r["wins"], "AP verdict": r["verdict"],
        "F1": run["F1"].mean(), "dF1": run["F1"].mean() - reference_run["F1"].mean()}
pd.set_option("display.width", 250); pd.set_option("display.max_colwidth", 120)
print(pd.DataFrame.from_dict(board, orient="index").round(4).to_string())

                                                                                                                 AP     dAP AP better on        AP verdict      F1     dF1
min_samples_leaf=20, max_samples=0.7, max_features=sqrt, max_depth=None, class_weight=None                   0.8492  0.0021          5/7  inside the noise  0.7603  0.0004
min_samples_leaf=20, max_samples=None, max_features=sqrt, max_depth=10, class_weight=balanced_subsample      0.8485  0.0015          4/7  inside the noise  0.7602  0.0004
min_samples_leaf=20, max_samples=None, max_features=0.3, max_depth=6, class_weight=balanced_subsample        0.8480  0.0009          4/7  inside the noise  0.7549 -0.0050
min_samples_leaf=10, max_samples=None, max_features=0.3, max_depth=6, class_weight=None                      0.8479  0.0009          3/7  inside the noise  0.7575 -0.0024
min_samples_leaf=5, max_samples=0.7, max_features=sqrt, max_depth=None, class_weight=balanced_subsample      0.8477  0.0006          4/7  inside 

**Conclusions:**
- The five best combinations are within 0.0015 of each other in AP (0.8477 to 0.8492, with 150 trees) and none is above the starting point (0.8471) by more than two standard errors. The best one (`min_samples_leaf=20`, `max_samples=0.7`) is +0.0021, ahead on 5 of 7 splits, inside the noise. Their F1 differs from the starting point by −0.0050 to +0.0004.
- The search is small (10 combinations out of 320, plus the starting point), but the pattern is flat: settings as different as no depth limit and a depth of 6 with 0.3 of the columns at each split land within 0.002 of each other in AP.

In [7]:
# For each split from the third on: the combination chosen on the earlier splits, scored on it
honest = walk_forward(scores, first=2)
on_same = reference_run.loc[honest["split"]]
dates_of_splits = [dates.iloc[rows_[1]].min().date() for rows_ in time_splits]
print(pd.DataFrame({
    "validation from": [dates_of_splits[s] for s in honest["split"]],
    "chosen on the earlier splits": [config_name(i) for i in honest["chosen"]],
    "AP": honest["AP"].to_numpy(), "AP start": on_same["AP"].to_numpy(),
    "F1": honest["F1"].to_numpy(), "F1 start": on_same["F1"].to_numpy()}).round(4).to_string(index=False))

search_vs_start = pd.DataFrame({m: paired_delta(honest[m].to_numpy(), on_same[m].to_numpy())
                                for m in ("AP", "F1")}).T
print("\nsearch minus starting point on the splits it did not see:\n", search_vs_start.to_string())

chosen = int(mean_ap.idxmax()) if search_vs_start.loc["AP", "verdict"] == "better" else default
final_settings = grid[chosen]
print(f"\nbest combination on all 7 splits: {config_name(int(mean_ap.idxmax()))}")
print(f"settings of the final model: {config_name(chosen)}")

validation from                                                                         chosen on the earlier splits     AP  AP start     F1  F1 start
     2021-03-06 min_samples_leaf=1, max_samples=None, max_features=0.5, max_depth=6, class_weight=balanced_subsample 0.8643    0.8664 0.7623    0.7611
     2021-11-13           min_samples_leaf=20, max_samples=0.7, max_features=sqrt, max_depth=None, class_weight=None 0.8260    0.8257 0.7354    0.7239
     2022-10-04           min_samples_leaf=20, max_samples=0.7, max_features=sqrt, max_depth=None, class_weight=None 0.8256    0.8154 0.7263    0.7315
     2023-07-29           min_samples_leaf=20, max_samples=0.7, max_features=sqrt, max_depth=None, class_weight=None 0.8775    0.8737 0.7932    0.7931
     2024-04-13           min_samples_leaf=20, max_samples=0.7, max_features=sqrt, max_depth=None, class_weight=None 0.8711    0.8737 0.8090    0.8175

search minus starting point on the splits it did not see:
        delta       SEM wins       

**Conclusions:**
- Chosen on the earlier splits only, the search picks one combination for the first split it scores (`min_samples_leaf=1`, `max_features=0.5`, `max_depth=6`, balanced class weights) and then the same one four times (`min_samples_leaf=20`, `max_samples=0.7`). On the 5 splits it did not see, it scores AP +0.0019 ± 0.0024 against the starting point (better on 3 of 5) and F1 −0.0002 ± 0.0034 (better on 3 of 5), both inside the noise.
- The search does not beat the starting point, so the final model keeps it: `min_samples_leaf=5`, `max_features="sqrt"`, no depth limit, no class weights and every row in each tree's bootstrap, with 1000 trees instead of 300.
- The settings of the forest move AP by a few thousandths, far less than the differences between periods, so they are not where the score can improve.

## 4.2.6 Final Model and Kaggle Submission

As in the baseline notebook, the final model repeats the steps of each split once, on every training row. `feature_selection_l1.csv` carries the training rows prepared by the recipe and reduced to the selected columns, and `feature_selection_l1_test.csv` the test rows prepared with the same fits. The forest has the settings chosen in 4.5 and, this time, **1000 trees**: a final model has no reason to keep the 300 used to compare combinations, and more trees only make the probabilities steadier. The cut-off is chosen on the out-of-bag probabilities of the training rows, and the submission keeps `ID` and writes 0/1 labels. This is version 02.

In [8]:
train_table = pd.read_csv("../../../data/featureselection/feature_selection_l1.csv")
test_table = pd.read_csv("../../../data/featureselection/feature_selection_l1_test.csv")
features = [c for c in train_table.columns if c not in ("ID", "customer_id", target)]
assert list(test_table.columns) == list(train_table.columns.drop(target))

final_model, _, final_cutoff = fit_forest_with_cutoff(
    make_forest(**{**final_settings, "n_estimators": 1000}), train_table[features], train_table[target])
test_probabilities = final_model.predict_proba(test_table[features])[:, 1]
submission = pd.DataFrame({"ID": test_table["ID"], target: (test_probabilities >= final_cutoff).astype(int)})

sample = pd.read_csv("../../../data/sample_submission.csv")
assert list(submission["ID"]) == list(sample["ID"]) and set(submission[target]) <= {0, 1}

submission_file = Path("../../../data/submissions/ML_Group_18_Version02.csv")
submission_file.parent.mkdir(exist_ok=True)
submission.to_csv(submission_file, index=False)

previous = pd.read_csv("../../../data/submissions/ML_Group_18_Version01.csv")
agreement = (previous[target].to_numpy() == submission[target].to_numpy()).mean()
print(f"{len(features)} features | cut-off {final_cutoff} | {submission[target].mean():.1%} of the"
      f" test rows predicted to come back | saved to {submission_file.as_posix()}")
print(f"agreement with the logistic regression of version 01: {agreement:.1%}")

40 features | cut-off 0.36 | 83.1% of the test rows predicted to come back | saved to ../../../data/submissions/ML_Group_18_Version02.csv
agreement with the logistic regression of version 01: 89.6%


**Conclusions:**
- The final forest has 40 columns, 1000 trees and a cut-off of 0.36, chosen on the out-of-bag probabilities of every training row. It predicts 83.1% of the 986 test IDs to come back, against 76.7% for the logistic regression of version 01, and agrees with it on 89.6% of the IDs.
- The test rows also look more likely to come back to the model: their mean predicted probability is 0.613, against 0.560 on the training rows. The cut-off is not a sharp choice: the out-of-bag F1 is almost the same at 0.36 and 0.40 (0.7655 and 0.7651), while the share of test rows predicted to come back goes from 83.1% to 79.2%.

## 4.2.7 Conclusions

- **Model:** a random forest with 1000 trees, `min_samples_leaf=5`, `max_features="sqrt"`, no depth limit and no class weights, on the 40 columns the feature-selection strategy keeps, with the cut-off chosen on the out-of-bag probabilities of the training rows (0.36 on every training row).
- **Assessed performance:** F1 **0.763 ± 0.014** and AP 0.849 over the 7 temporal splits, from 0.723 to 0.823 in F1 depending on the period, against 0.754 and 0.842 for the logistic regression, 0.746 for the baseline rule and 0.719 for always 1. The forest is ahead of the logistic regression and of the rule on all 7 splits, by +0.0092 and +0.0173 in F1, beyond two standard errors. This is the performance the Kaggle result should be read against.
- **Feature selection:** not needed by the forest: with all columns (83 to 106) AP and F1 stay inside the noise of the selected columns.
- **Tuning:** a random search of 10 combinations plus the starting point, scored on splits that took no part in the choice, does not beat the starting point (AP +0.0019, F1 −0.0002, both inside the noise).
- **Submission:** `ML_Group_18_Version02.csv` holds the 986 test IDs; 83.1% of them are predicted to come back, against 76.7% in version 01.
- **Limits:** the recipe, the feature-selection strategy and the cut-off rule were all decided on these same 7 splits, so the estimate is slightly optimistic. The forest's advantage over the logistic regression is small next to the differences between periods. The random search covers a small part of the space. Only two models are assessed here; the final project compares at least five, with their hyperparameters tuned inside the temporal splits.